# Cooperation for compounds from other members: consumption overlap

In [519]:
import pandas as pd
import json
import seaborn as sns
import numpy as np

import networkx as nx

from pyvis.network import Network
import matplotlib.pyplot as plt
import sys

sys.path.append("../functions/")
import networks

In [520]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [521]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = media_db[media_db.medium=="M9"].compound.values
compounds = list(compounds)
compounds.remove("glc__D")
m9_compounds = ["M_"+met+"__91__e__93__" for met in compounds]

In [522]:
with open("../data/Venturelli2018/generated/metid2metname.json") as json_file:
    metid2metname = json.load(json_file)

results = pd.read_csv("../output/Venturelli2018/results_steadiercom/NONE_abundance.tsv",sep="\t")

results["mass_rate*frequency"] = results["frequency"]*results["mass_rate"]
results = results[~results.compound.isin(m9_compounds)]

**Use the member that consumes the least as a threshold**

In [523]:
receiving = results.groupby(["community","receiver"]).sum()["mass_rate*frequency"].sort_values()
print(f"Consumption (mass_rate) of the member that consumes the least: {receiving.min()}")

Consumption (mass_rate) of the member that consumes the least: 0.00048505482503399496


In [524]:
print(f"1% of this is: {receiving.min()*0.01}")

1% of this is: 4.85054825033995e-06


**Set thresholds**

In [541]:
results = results[results.frequency>0.01]
results = results[results["mass_rate*frequency"]>1e-6] ## From above

## With dataset 4 data

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Closeness centrality**: The reciprocal of sum of shortest paths to i (Incoming path for directed graph)
- **Degree centrality**: The fraction of nodes that node i is connected to. (Sum of incoming and outgoing edges for directed graphs)
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes (right eigenvector centrality is calculated)


In [526]:
import matplotlib as mpl
# Color end points for colorFader
b1='#f8cdcf' #bright pink
b2='#fa4f56' #red

c1='#ffffff' #white
c2='#0b5394' #blue
    
## From https://stackoverflow.com/questions/25668828/how-to-create-colour-gradient-in-python
def colorFader(c1,c2,mix=0): #fade (linear interpolate) from color c1 (at mix=0) to c2 (mix=1)
    c1=np.array(mpl.colors.to_rgb(c1))
    c2=np.array(mpl.colors.to_rgb(c2))
    return mpl.colors.to_hex((1-mix)*c1 + mix*c2)





In [527]:
def cooperation_network(network_data,network_exp,closeness_centrality_met,cutoff=0,use_count=True):
    net = Network(directed=True)
    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    
    if isinstance(closeness_centrality_met,pd.Series):
        maximum_centrality = closeness_centrality_met.max()
        color = [colorFader(b1,b2,closeness_centrality_met[node]/maximum_centrality) for node in nodes]
        value = [(closeness_centrality_met[node]*10) for node in nodes]

        net.add_nodes(nodes,color=color,value=value)
    else:
        net.add_nodes(nodes)
    

    maximum_overlap = network_data.max()
    
    i = 0
    j = 0
    for index,value in network_data.items():
        new_index1 = index[0].split("_")[0][0].upper()+index[0].split("_")[1][0].upper()
        new_index2 = index[1].split("_")[0][0].upper()+index[1].split("_")[1][0].upper()
        
        new_index = (new_index1,new_index2)
        if value>cutoff:
            
            
            if new_index in network_exp.index:
                color = "red"
                i+=1
                
            else:
                color = colorFader(c1,c2,value/maximum_overlap)
            if use_count:
                
                net.add_edge(index[0],index[1],width=value*2,color=color)

            else:
                net.add_edge(index[0],index[1],width=value*10,color=color) 
            j+=1
    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)
    print(f"Matching exp for {(i/j)*100}% of interactions")

    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net

## Cooperation based on COUNT (Only metabolites from metabolomics + freq_cutoff = 0.1 !!)

In [528]:
dataset4 = pd.read_excel("../data/Venturelli2018/msb178157-sup-0005-datasetev4.xlsx",sheet_name="Dataset EV4",index_col=0)

#### Use count (freq_cutoff=0.1,metabolomics_mets_only=True)

In [529]:
consumption = dataset4<=-3
consumption = consumption.map(int)
consumption.name="consumption"
consumption.columns = consumption.columns.map(lambda x: x.replace("'",""))
consumption.columns = consumption.columns.map(lambda x: x.replace(" ",""))

production = dataset4>=1
production = production.map(int)
production.name="production"
production.columns = production.columns.map(lambda x: x.replace("'",""))
production.columns = production.columns.map(lambda x: x.replace(" ",""))
network_exp = production.T.dot(consumption)

network_exp = network_exp.stack()
network_exp= network_exp[network_exp>0]

In [530]:

network_data = networks.cooperation_network_data(results,metabolomics_mets_only=True,use_count=True,freq_cutoff=0.1)

centrality_metrics_SP = networks.cooperation_centrality_metrics(network_data).sort_values("closeness (1/data)")


#closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"]

net = cooperation_network(network_data,network_exp,closeness_centrality_met=False,use_count=True)
net.show("network_file.html")

Matching exp for 34.78260869565217% of interactions
network_file.html


#### Use count (freq_cutoff=0.1,metabolomics_mets_only=True)

In [531]:
consumption = dataset4<=-1
consumption = consumption.map(int)
consumption.name="consumption"
consumption.columns = consumption.columns.map(lambda x: x.replace("'",""))
consumption.columns = consumption.columns.map(lambda x: x.replace(" ",""))

production = dataset4>=1
production = production.map(int)
production.name="production"
production.columns = production.columns.map(lambda x: x.replace("'",""))
production.columns = production.columns.map(lambda x: x.replace(" ",""))
network_exp = production.T.dot(consumption)

network_exp = network_exp.stack()
network_exp= network_exp[network_exp>0]

In [532]:

network_data = networks.cooperation_network_data(results,metabolomics_mets_only=True,use_count=True,freq_cutoff=0.1)

centrality_metrics_SP = networks.cooperation_centrality_metrics(network_data).sort_values("closeness (1/data)")


closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"]

net = cooperation_network(network_data,network_exp,closeness_centrality_met=False,use_count=True)
net.save_graph("figures/network_member_member_count_cooperation.html")
net.show("network_file.html")

Matching exp for 43.47826086956522% of interactions
network_file.html


## Cooperation based on FREQUENCY (All metabolites + freq_cutoff = 0.1 !!)

In [533]:

network_data = networks.cooperation_network_data(results,metabolomics_mets_only=True,use_count=False,freq_cutoff=0.1)

In [534]:
float(network_data.min())

0.104

In [535]:
float(network_data.max())

1.0

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Closeness centrality**: The reciprocal of sum of shortest paths to i (Incoming path for directed graph)
- **Degree centrality**: The fraction of nodes that node i is connected to. (Sum of incoming and outgoing edges for directed graphs)
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes (right eigenvector centrality is calculated)


In [536]:
dataset4 = pd.read_excel("../data/Venturelli2018/msb178157-sup-0005-datasetev4.xlsx",sheet_name="Dataset EV4",index_col=0)


In [537]:
consumption = dataset4<=-3
consumption = consumption.map(int)
consumption.name="consumption"
consumption.columns = consumption.columns.map(lambda x: x.replace("'",""))
consumption.columns = consumption.columns.map(lambda x: x.replace(" ",""))

production = dataset4>=1
production = production.map(int)
production.name="production"
production.columns = production.columns.map(lambda x: x.replace("'",""))
production.columns = production.columns.map(lambda x: x.replace(" ",""))
network_exp = production.T.dot(consumption)

network_exp = network_exp.stack()
network_exp= network_exp[network_exp>0]

In [538]:
net = cooperation_network(network_data,network_exp,closeness_centrality_met=closeness_centrality_met,cutoff=0.1,use_count=False)
net.show("network_file.html")

Matching exp for 34.78260869565217% of interactions
network_file.html


In [539]:
consumption = dataset4<=-1
consumption = consumption.map(int)
consumption.name="consumption"
consumption.columns = consumption.columns.map(lambda x: x.replace("'",""))
consumption.columns = consumption.columns.map(lambda x: x.replace(" ",""))

production = dataset4>=1
production = production.map(int)
production.name="production"
production.columns = production.columns.map(lambda x: x.replace("'",""))
production.columns = production.columns.map(lambda x: x.replace(" ",""))
network_exp = production.T.dot(consumption)

network_exp = network_exp.stack()
network_exp= network_exp[network_exp>0]

In [540]:
net = cooperation_network(network_data,network_exp,closeness_centrality_met=closeness_centrality_met,cutoff=0.1,use_count=False)
net.show("network_file.html")

Matching exp for 43.47826086956522% of interactions
network_file.html
